In [1]:
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/sample_submission.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/dataset-metadata.json
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/baseline_submission.csv
/kaggle/input/models/keras/sentence-transformers/keras/all_minilm_l12_v2_en/1/config.json
/kaggle/input/models/keras/sentence-transformers/keras/all_minilm_l12_v2_en/1/preprocessor.json
/kaggle/input/models/keras/sentence-transformers/keras/all_minilm_l12_v2_en/1/tokenizer.json
/kaggle/input/

# **Importing Relevant Libraries** 

In [5]:
!pip install -q rank-bm25

import re
import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sentence_transformers import SentenceTransformer, util
 
device = "cuda" if torch.cuda.is_available() else "cpu"
 
base = "/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/"

# **Data Preprocessing**

In [6]:
docs = pd.read_csv(base + "documents.csv")
docs["title"] = docs["title"].fillna("")
docs["text"] = docs["text"].fillna("")
docs["content"] = docs["title"] + ". " + docs["text"]

train_queries = pd.read_csv(base + "train_queries.csv")
qrels_train = pd.read_csv(base + "qrels_train.csv")
test = pd.read_csv(base + "test_queries.csv")

In [9]:
#  TOKENIZing TEXT

def tokenize(text):
    tokens = re.findall(r'\b[a-zA-Z]{2,}\b', str(text).lower())
    return [t for t in tokens if t not in ENGLISH_STOP_WORDS]


#  BUILDING BM25 INDEX 
doc_tokens = [tokenize(doc) for doc in docs["content"]]
bm25 = BM25Okapi(doc_tokens, k1=1.2, b=0.75)  # standard defaults; tune later using local eval


# BUILDING DENSE EMBEDDINGS
dense_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=device)
instruction = "Represent this sentence for searching relevant passages: "

doc_embeddings = dense_model.encode(docs["content"].tolist(), convert_to_tensor=True, batch_size=32)


def min_max_scale(scores):
    s_min, s_max = scores.min(), scores.max()
    return np.zeros_like(scores) if s_max == s_min else (scores - s_min) / (s_max - s_min)


def hybrid_retrieve_top5(query_str, doc_embeddings, alpha=0.6):
    """Returns the top 5 document indices (not IDs) for one query,
    using a hybrid of BM25 (sparse) and dense embedding similarity."""
    bm25_sc = bm25.get_scores(tokenize(query_str))

    query_embedding = dense_model.encode([instruction + query_str], convert_to_tensor=True)
    dense_sc = util.cos_sim(query_embedding, doc_embeddings).cpu().numpy()[0]

    hybrid_score = alpha * min_max_scale(dense_sc) + (1 - alpha) * min_max_scale(bm25_sc)
    top5_idx = np.argsort(hybrid_score)[::-1][:5]
    return top5_idx

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [11]:
# EVALUATION (nDCG@5) 
def dcg_at_5(relevances):
    return sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevances))


def ndcg_at_5(query_id, predicted_doc_ids, qrels_df):
    relevant = qrels_df[qrels_df["query_id"] == query_id]
    rel_lookup = dict(zip(relevant["document_id"], relevant["relevance"]))

    predicted_rels = [rel_lookup.get(doc_id, 0) for doc_id in predicted_doc_ids]
    dcg = dcg_at_5(predicted_rels)

    ideal_rels = sorted(rel_lookup.values(), reverse=True)[:5]
    idcg = dcg_at_5(ideal_rels)

    return dcg / idcg if idcg > 0 else 0.0


scores = []
for _, row in train_queries.iterrows():
    top5_idx = hybrid_retrieve_top5(row["query"], doc_embeddings, alpha=0.6)
    predicted_doc_ids = [int(docs.iloc[i]["document_id"]) for i in top5_idx]
    score = ndcg_at_5(row["query_id"], predicted_doc_ids, qrels_train)
    scores.append(score)

print("Hybrid (BM25 + dense) average nDCG@5 on TRAINING data:", np.mean(scores))

Hybrid (BM25 + dense) average nDCG@5 on TRAINING data: 0.7363234663182979


**SUBMISSION**

In [12]:
rows = []
for _, row in test.iterrows():
    qid = row["query_id"]
    top5_idx = hybrid_retrieve_top5(row["query"], doc_embeddings, alpha=0.6)

    for i in top5_idx:
        rows.append({
            "QueryId": qid,
            "DocumentId": int(docs.iloc[i]["document_id"]),
        })

submission_df = pd.DataFrame(rows)
submission_df.to_csv("submission.csv", index=False)
print("Saved submission.csv successfully!")
print(submission_df.head(10))

Saved submission.csv successfully!
   QueryId  DocumentId
0     1001           1
1     1001           3
2     1001           4
3     1001           2
4     1001           5
5     1002           1
6     1002           3
7     1002           4
8     1002           5
9     1002           2


the score isn't impressive

In [ ]:
# TRYING A CROSS-ENCODER RE-RANKER 
# =========================================================
reranker = CrossEncoder("BAAI/bge-reranker-large", max_length=512, device=device)
 
TOP_K_FIRST_STAGE = 30  # candidate pool size before re-ranking
ALPHA = 0.6             # weight on dense score vs BM25 in stage 1
 
 
def two_stage_retrieve_top5(query_str):
   

    bm25_sc = bm25.get_scores(tokenize(query_str))
 
    query_embedding = dense_model.encode([instruction + query_str], convert_to_tensor=True)
    dense_sc = util.cos_sim(query_embedding, doc_embeddings).cpu().numpy()[0]
 
    hybrid_score = ALPHA * min_max_scale(dense_sc) + (1 - ALPHA) * min_max_scale(bm25_sc)
    candidate_indices = np.argsort(hybrid_score)[::-1][:TOP_K_FIRST_STAGE]
 
    # --- Stage 2: cross-encoder re-ranking ---
    pairs = [(query_str, docs.iloc[c_idx]["content"]) for c_idx in candidate_indices]
    rerank_scores = reranker.predict(pairs)
 
    top5_local_idx = np.argsort(rerank_scores)[::-1][:5]
    top5_doc_indices = [candidate_indices[i] for i in top5_local_idx]
    return top5_doc_indices
 
 

def dcg_at_5(relevances):
    return sum(rel / np.log2(idx + 2) for idx, rel in enumerate(relevances))
 
 
def ndcg_at_5(query_id, predicted_doc_ids, qrels_df):
    relevant = qrels_df[qrels_df["query_id"] == query_id]
    rel_lookup = dict(zip(relevant["document_id"], relevant["relevance"]))
 
    predicted_rels = [rel_lookup.get(doc_id, 0) for doc_id in predicted_doc_ids]
    dcg = dcg_at_5(predicted_rels)
 
    ideal_rels = sorted(rel_lookup.values(), reverse=True)[:5]
    idcg = dcg_at_5(ideal_rels)
 
    return dcg / idcg if idcg > 0 else 0.0
 

 
# This loop calls the reranker once per training query, which is slower
# than the hybrid-only version
scores = []
for _, row in train_queries.iterrows():
    top5_idx = two_stage_retrieve_top5(row["query"])
    predicted_doc_ids = [int(docs.iloc[i]["document_id"]) for i in top5_idx]
    score = ndcg_at_5(row["query_id"], predicted_doc_ids, qrels_train)
    scores.append(score)
 
print("Two-stage (hybrid + re-ranker) average nDCG@5 on TRAINING data:", np.mean(scores))

In [ ]:

# GENERATING FINAL TEST SUBMISSION

rows = []
for _, row in test.iterrows():
    qid = row["query_id"]
    top5_idx = two_stage_retrieve_top5(row["query"])
 
    for i in top5_idx:
        rows.append({
            "QueryId": qid,
            "DocumentId": int(docs.iloc[i]["document_id"]),
        })
 
submission_df = pd.DataFrame(rows)
submission_df.to_csv("submission.csv", index=False)
print("Saved submission.csv successfully!")
print(submission_df.head(10))